# Lesson 7 — Checkpoints

**You will:** pause an agent run for human approval, then resume it. Build an email-reply agent that drafts but won't send.

[Open in Colab](https://colab.research.google.com/github/richey-malhotra/barebear/blob/main/lessons/07-checkpoints/lesson.ipynb)
[Read the lesson narrative](./lesson.md)
[Back to syllabus](../README.md)

## The big idea

Some actions are too consequential to be fully autonomous. Sending an email. Approving a refund. Deploying code. For these, you want the agent to do the *work* (read the ticket, draft the reply) — but stop, hand control to a human, and only proceed if approved.

That's a **checkpoint**. When the agent tries to call a tool that requires approval, the run pauses cleanly. A `Checkpoint` object captures everything needed to resume. Later, a human approves or rejects, and you call `bear.resume()` to continue.

This is the pattern for *trustworthy* agentic systems in production. Agent does 90%. Human signs off on the 10% that matters.

## Setup

Run the next two cells once per session. They install barebear and set your OpenRouter key (free tier — get one at <https://openrouter.ai>).

In [ ]:
%pip install -q "barebear[openai]"

In [ ]:
import os
from getpass import getpass

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("Paste your OpenRouter key (sk-or-...): ")

# Optional: pin a specific free-tier model. If unset, barebear uses its default.
# Free-tier model availability rotates — swap here if a lesson cell errors with
# "model not available". Any *:free model on https://openrouter.ai/models works.
# os.environ["BAREBEAR_MODEL"] = "meta-llama/llama-3.2-3b-instruct:free"

print("Key set. First 8 characters:", os.environ["OPENROUTER_API_KEY"][:8] + "...")
print("Model:", os.environ.get("BAREBEAR_MODEL", "(framework default)"))

## An email-reply agent that pauses before sending

In [ ]:
from barebear import Bear, Task, Tool, Policy, OpenRouterModel

def read_ticket(ticket_id: str) -> str:
    return f"Ticket {ticket_id} from dana@example.com: 'Reset link expired, locked out, deadline in 2 hours, please help.'"

def draft_email(to: str, subject: str, body: str) -> str:
    return f"DRAFT:\n  To: {to}\n  Subject: {subject}\n  Body: {body[:200]}..."

def send_email(to: str, subject: str, body: str) -> str:
    return f"Email sent to {to}."

bear = Bear(
    model=OpenRouterModel(),
    tools=[
        Tool(name="read_ticket", fn=read_ticket, description="Read a support ticket by ID"),
        Tool(name="draft_email", fn=draft_email, description="Draft an email (no side effects)"),
        Tool(name="send_email", fn=send_email, description="Send an email to a customer",
             requires_approval=True, side_effects="external", risk="high"),
    ],
    policy=Policy(require_approval_for=["send_email"]),
)

report = bear.run(Task(goal="Read ticket TK-42 and reply with help."), trace=True)
print()
print("status:", report.status)
print("checkpoint id:", report.checkpoint_id)

## Inspect the checkpoint

The run paused. Now you can read what the agent *wanted* to do, decide whether you'd approve, and resume.

In [ ]:
cp = bear.checkpoints.get(report.checkpoint_id)
print("pending action:", cp.pending_action)

## Approve and resume

In [ ]:
final = bear.resume(cp, approved=True)
print(final.summary())

## Reject and resume

In a fresh run, try rejecting instead. The agent finishes with `status='failed'` and an error noting the rejection.

In [ ]:
report2 = bear.run(Task(goal="Read ticket TK-42 and reply with help."))
if report2.status == "paused":
    cp2 = bear.checkpoints.get(report2.checkpoint_id)
    final2 = bear.resume(cp2, approved=False)
    print(final2.summary())

## Exercise

1. Serialise a checkpoint to JSON between pause and resume — `cp.to_dict()` if available, otherwise inspect the fields manually. Confirm you could reconstruct the run from disk after a long delay.
2. Build an agent with **two** approval-required tools (`send_email` and `issue_refund`). Watch the order of pauses.

## What's next

[Lesson 8 →](../08-planning/lesson.ipynb)